# When Arrivals Aren't Independent: Copulas in SimWeave

Companion notebook for the [SimWeave blog post](https://notabot123.github.io/simweave/blog/copulas-dependence/).

This notebook covers:
- Joint draws from a copula with chosen marginals (`JointSampler`)
- Tail dependence and why the copula family matters
- A queue with correlated gap and service time (`ArrivalGenerator(joint=...)`)
- Bursty arrivals with identical marginals (`MarkovCopulaProcess`)
- Fitting a copula to data (`fit_copula`)

It needs `simweave>=0.9.0`; the plots use the `viz` extra (plotly). Everything except the Student-t copula runs without scipy.

In [1]:
# Uncomment to install if running outside Binder
# !pip install 'simweave[viz,optim]>=0.9.0'

import numpy as np
import plotly.graph_objects as go
import simweave as sw
from simweave.stats import Exponential, LogNormal, JointSampler, MarkovCopulaProcess, kendall_tau

print("simweave", sw.__version__)

simweave 0.9.0


## 1  A first joint draw

Pick the dependence (a Gumbel copula with Kendall tau 0.6) and the marginals (exponential gaps, lognormal sizes) separately.

In [2]:
rng = np.random.default_rng(42)
joint = JointSampler(
    sw.GumbelCopula.from_kendall_tau(0.6),
    {"interarrival": Exponential(1.0), "size": LogNormal.from_mean_cv(2.0, 0.5)},
)
x = joint.sample(rng, 50_000)
print("mean gap", round(x["interarrival"].mean(), 3), "mean size", round(x["size"].mean(), 3))
print("kendall tau", round(kendall_tau(x["interarrival"][:10_000], x["size"][:10_000]), 3))

mean gap 1.002 mean size 1.999


kendall tau 0.595


## 2  Dependence has a shape

Same Kendall tau of 0.5, same exponential marginals, three tail behaviours.

In [3]:
cops = {
    "Gaussian": sw.GaussianCopula.from_kendall_tau(0.5),
    "Clayton (lower tail)": sw.ClaytonCopula.from_kendall_tau(0.5),
    "Gumbel (upper tail)": sw.GumbelCopula.from_kendall_tau(0.5),
}
for name, c in cops.items():
    td = c.tail_dependence()
    print(f"{name:22s} lower {td['ll']:.2f}  upper {td['uu']:.2f}")

m = Exponential(1.0)
fig = go.Figure()
for name, c in cops.items():
    u = c.sample_u(2000, np.random.default_rng(7))
    fig.add_trace(go.Scatter(x=m.ppf(u[:, 0]), y=m.ppf(u[:, 1]), mode="markers",
                             marker=dict(size=4, opacity=0.4), name=name))
fig.update_layout(title="Same marginals, same tau, different dependence",
                  xaxis=dict(title="gap", range=[0, 7]), yaxis=dict(title="service time", range=[0, 7]))
fig.show()

Gaussian               lower 0.00  upper 0.00
Clayton (lower tail)   lower 0.71  upper 0.00
Gumbel (upper tail)    lower 0.00  upper 0.59


Negative dependence, or putting the tail on the other corner, is a reflection of an axis:

In [4]:
g = sw.GumbelCopula.from_kendall_tau(0.5)
neg = g.flipped((1,))
print(neg.kendall_tau())
print(neg.tail_dependence())

-0.5
{'ll': 0.0, 'uu': 0.0, 'lu': 0.0, 'ul': 0.5857864376269049}


## 3  Correlated gap and service time in a queue

`ArrivalGenerator(joint=...)` hands the whole draw to the factory; `EntityProperties.from_draw` turns the drawn service time into the entity's service time.
`gap_alignment="after"` pairs a job with the gap that *follows* it.

In [5]:
def factory(env, draw):
    job = sw.Entity()
    job.sim_properties = sw.EntityProperties.from_draw(draw)
    return job


def time_in_system(copula, seed, sim_time=2000.0):
    rng = np.random.default_rng(seed)
    env = sw.SimEnvironment(dt=0.05)
    service = sw.Service(capacity=1, buffer_size=100_000, rng=rng)
    joint = JointSampler(copula, {"interarrival": Exponential(1.0), "service_time": Exponential(0.8)})
    env.register(sw.ArrivalGenerator(None, factory, service, rng=rng, joint=joint, gap_alignment="after"))
    env.register(service)
    env.run(sim_time)
    return service.average_residence()


candidates = {
    "independent": sw.GaussianCopula(1e-9),
    "gaussian": sw.GaussianCopula.from_kendall_tau(-0.5),
    "clayton (flipped)": sw.ClaytonCopula.from_kendall_tau(0.5).flipped((0,)),
    "gumbel (flipped)": sw.GumbelCopula.from_kendall_tau(0.5).flipped((0,)),
}
results = {}
for name, cop in candidates.items():
    runs = np.array([time_in_system(cop, s) for s in range(6)])
    results[name] = runs
    print(f"{name:20s} {runs.mean():5.2f} +/- {1.96 * runs.std(ddof=1) / np.sqrt(len(runs)):.2f}  (95% CI)")

independent           4.40 +/- 0.36  (95% CI)


gaussian              6.62 +/- 1.03  (95% CI)


clayton (flipped)     5.66 +/- 0.75  (95% CI)


gumbel (flipped)      6.47 +/- 1.55  (95% CI)


Dependence moves the answer a lot. The family matters much less at this load, and the intervals overlap. Try more replicates, heavier-tailed service times or a finite buffer to look for tail effects.

## 4  Bursty arrivals with identical marginals

`MarkovCopulaProcess` makes a sequence with the marginal you choose and lag-1 dependence from the copula. It drops into the `interarrival` slot.

In [6]:
proc = MarkovCopulaProcess(sw.GumbelCopula.from_kendall_tau(0.6), Exponential(1.0))
gaps = proc.sample(np.random.default_rng(3), 20_000)
print("mean", round(gaps.mean(), 3), "lag-1 tau", round(kendall_tau(gaps[:-1], gaps[1:]), 3))
short = gaps[:-1] < 0.2
print("P(short gap next | short gap now)", round(float(np.mean(gaps[1:][short] < 0.2)), 3),
      "vs overall", round(float(np.mean(gaps < 0.2)), 3))

fig = go.Figure(go.Scatter(x=gaps[:-1][:2500], y=gaps[1:][:2500], mode="markers",
                           marker=dict(size=4, opacity=0.4)))
fig.update_layout(title="Each gap against the next", xaxis_title="gap t", yaxis_title="gap t+1")
fig.show()

mean 0.985 lag-1 tau 0.604
P(short gap next | short gap now) 0.594 vs overall 0.192


In [7]:
def queue_with_gaps(dependent, seed, sim_time=2000.0):
    rng = np.random.default_rng(seed)
    env = sw.SimEnvironment(dt=0.05)
    service = sw.Service(capacity=1, buffer_size=100_000, rng=rng)
    gaps = (MarkovCopulaProcess(sw.GumbelCopula.from_kendall_tau(0.6), Exponential(1.0))
            if dependent else sw.exponential(1.0))   # build per replicate: the process holds state

    def make(e):
        item = sw.Entity()
        item.sim_properties = sw.EntityProperties(service_time=sw.exponential(0.8))
        return item

    env.register(sw.ArrivalGenerator(gaps, make, service, rng=rng))
    env.register(service)
    env.run(sim_time)
    return service.average_residence()


for label, dep in (("independent", False), ("serially dependent", True)):
    runs = np.array([queue_with_gaps(dep, s) for s in range(6)])
    print(f"{label:20s} time in system {runs.mean():5.2f} +/- {1.96 * runs.std(ddof=1) / np.sqrt(len(runs)):.2f}")

independent          time in system  4.33 +/- 0.53


serially dependent   time in system 20.87 +/- 3.30


## 5  Fitting a copula to data

`fit_copula` uses ranks only, so the marginals do not matter. It ranks families by AIC (lower is better).

In [8]:
data = sw.GumbelCopula.from_kendall_tau(0.6).flipped((1,)).sample_u(3000, np.random.default_rng(1))
fit = sw.fit_copula(data)
print(fit.family, round(fit.tau, 3))
for row in fit.table[:4]:
    print(row["family"], round(row["aic"], 1))

gumbel (axis 1 flipped) -0.606
gumbel -3420.1
t -3191.7
clayton_survival -3114.5
gaussian -3067.8


In [9]:
# From observations straight to a simulation input (empirical marginals + fitted copula):
observations = JointSampler(
    sw.GumbelCopula.from_kendall_tau(0.6),
    {"interarrival": Exponential(1.0), "size": LogNormal.from_mean_cv(2.0, 0.5)},
).sample_array(np.random.default_rng(9), 5000)

fitted = JointSampler.from_data(observations, ["interarrival", "size"])
y = fitted.sample(np.random.default_rng(1), 20_000)
print("mean gap", round(y["interarrival"].mean(), 3), "tau", round(kendall_tau(y["interarrival"][:10_000], y["size"][:10_000]), 3))

mean gap 0.995 tau 0.587


## Where to go next

- Demos `30_serial_dependence_queue.py` and `31_copula_families_queue.py` in the library repo.
- The [Copula sampling design note](https://notabot123.github.io/simweave/design/copulas/) for what is planned next.